In [29]:
import pandas as pd
import numpy as np
from churn_prediction.paths import PROCESSED_ONLINE_RETAIL_DIR, FEATURES_ONLINE_RETAIL_DIR, INTERIM_CLI_ONLINE_RETAIL_DIR
import warnings
warnings.filterwarnings('ignore')

In [31]:
# load data
df = pd.read_parquet(PROCESSED_ONLINE_RETAIL_DIR / 'online_retail_clean.parquet')

# Pre-compute Revenue once, dùng chung cho tất cả nhóm
df['Revenue'] = df['Quantity'] * df['Price']
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

print(f"Data shape: {df.shape}")
print(f"Snapshot date: {snapshot_date.date()}")

Data shape: (779425, 9)
Snapshot date: 2011-12-10


### Nhóm 1: RFM
- `recency`: Số ngày từ lần mua cuối đến ngày snapshot
- `frequency`: Số hóa đơn đã mua
- `monetary`: Tổng số tiền đã chi
- `avg_order_value`: Giá trị trung bình mỗi hóa đơn
- `avg_items_per_order`: Số lượng sản phẩm trung bình mỗi đơn

In [32]:
rfm = (
    df
    .groupby('Customer ID')
    .agg(
        Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
        Frequency=('Invoice', 'nunique'),
        Monetary=('Revenue', 'sum'),
    )
    .reset_index()
)

# avg_order_value và avg_items_per_order cần group theo Invoice trước
order_level = (
    df
    .groupby(['Customer ID', 'Invoice'])
    .agg(
        order_revenue=('Revenue', 'sum'),
        order_items=('Quantity', 'sum'),
    )
    .reset_index()
)

order_agg = (
    order_level
    .groupby('Customer ID')
    .agg(
        avg_order_value=('order_revenue', 'mean'),
        avg_items_per_order=('order_items', 'mean'),
    )
    .reset_index()
)

rfm = rfm.merge(order_agg, on='Customer ID', how='left')

print(f"rfm: {rfm.shape[0]}")
rfm.head()

rfm: 5878


,Customer ID,Recency,Frequency,Monetary,avg_order_value,avg_items_per_order
0,12346.0,326,12,77556.46,6463.038333,6190.416667
1,12347.0,2,8,4921.53,615.191250,370.875000
2,12348.0,75,5,2019.40,403.880000,542.800000
3,12349.0,19,4,4428.69,1107.172500,406.000000
4,12350.0,310,1,334.40,334.400000,197.000000


### Nhóm 2: Hành vi mua hàng (Purchase Behavior Features)
- `total_quantity`: Tổng Quantity
- `total_orders`: Số Invoice duy nhất
- `unique_products`: Số StockCode khác nhau
- `unique_purchase_days`: Số ngày có giao dịch
- `avg_quantity_per_order`: total_quantity / total_orders
- `max_order_value`: Đơn hàng lớn nhất
- `min_order_value`: Đơn hàng nhỏ nhất

In [34]:
purchase_behavior = (
    df
    .groupby('Customer ID')
    .agg(
        total_quantity=('Quantity', 'sum'),
        total_orders=('Invoice', 'nunique'),
        unique_products=('StockCode', 'nunique'),
        unique_purchase_days=('InvoiceDate', lambda x: x.dt.date.nunique()),
    )
    .reset_index()
)

purchase_behavior['avg_quantity_per_order'] = (
    purchase_behavior['total_quantity'] / purchase_behavior['total_orders']
)

# max/min order value cần tính ở cấp Invoice
order_value = (
    order_level  # tái dùng từ Nhóm 1
    .groupby('Customer ID')
    .agg(
        max_order_value=('order_revenue', 'max'),
        min_order_value=('order_revenue', 'min'),
    )
    .reset_index()
)

purchase_behavior = purchase_behavior.merge(order_value, on='Customer ID', how='left')

print(f"Purchase Behavior: {purchase_behavior.shape[0]}")
purchase_behavior.head()

Purchase Behavior: 5878


,Customer ID,total_quantity,total_orders,unique_products,unique_purchase_days,avg_quantity_per_order,max_order_value,min_order_value
0,12346.0,74285,12,27,8,6190.416667,77183.60,1.00
1,12347.0,2967,8,126,8,370.875000,1294.32,224.82
2,12348.0,2714,5,25,5,542.800000,892.80,222.16
3,12349.0,1624,4,138,4,406.000000,1757.55,200.00
4,12350.0,197,1,17,1,197.000000,334.40,334.40


### Nhóm 3: Time-based Features
- `customer_age_days`: Từ lần mua đầu tiên đến snapshot
- `days_since_last_purchase`: Khoảng cách từ lần mua cuối
- `purchase_span_days`: Lần mua cuối - lần mua đầu
- `avg_days_between_orders`: Khoảng cách trung bình giữa các đơn

In [35]:
time_based = (
    df
    .groupby('Customer ID')
    .agg(
        first_date=('InvoiceDate', 'min'),
        last_date=('InvoiceDate', 'max'),
    )
    .reset_index()
)

time_based['customer_age_days'] = (
    (snapshot_date - time_based['first_date']).dt.days
)
time_based['days_since_last_purchase'] = (
    (snapshot_date - time_based['last_date']).dt.days
)
time_based['purchase_span_days'] = (
    (time_based['last_date'] - time_based['first_date']).dt.days
)

# avg_days_between_orders: chỉ tính được khi khách mua >= 2 lần
def avg_days_between(dates):
    sorted_dates = dates.sort_values().drop_duplicates()
    if len(sorted_dates) < 2:
        return np.nan
    return sorted_dates.diff().dt.days.mean()

avg_gap = (
    df
    .groupby('Customer ID')['InvoiceDate']
    .apply(avg_days_between)
    .reset_index()
    .rename(columns={'InvoiceDate': 'avg_days_between_orders'})
)

time_based = time_based.merge(avg_gap, on='Customer ID', how='left')
time_based = time_based.drop(columns=['first_date', 'last_date'])

print(f"time based: {time_based.shape[0]}")
time_based.head()

time based: 5878


,Customer ID,customer_age_days,days_since_last_purchase,purchase_span_days,avg_days_between_orders
0,12346.0,726,326,400,35.909091
1,12347.0,404,2,402,57.000000
2,12348.0,438,75,362,90.500000
3,12349.0,589,19,570,189.666667
4,12350.0,310,310,0,NaN


### Nhóm 4: Doanh thu (Revenue Features)
- `total_revenue`: Tổng doanh thu
- `avg_revenue_per_item`: Revenue / Quantity
- `revenue_std`: Độ biến động giá trị đơn hàng
- `max_revenue`: Đơn hàng lớn nhất
- `recent_30d_revenue`: Doanh thu 30 ngày gần nhất

In [36]:
revenue_features = (
    df
    .groupby('Customer ID')
    .agg(
        total_revenue=('Revenue', 'sum'),
        total_quantity_rev=('Quantity', 'sum'),
    )
    .reset_index()
)

revenue_features['avg_revenue_per_item'] = (
    revenue_features['total_revenue'] / revenue_features['total_quantity_rev']
)
revenue_features = revenue_features.drop(columns=['total_quantity_rev'])

# revenue_std và max_revenue tính ở cấp Invoice
rev_stats = (
    order_level
    .groupby('Customer ID')
    .agg(
        revenue_std=('order_revenue', 'std'),
        max_revenue=('order_revenue', 'max'),
    )
    .reset_index()
)

# recent_30d_revenue
cutoff_30d = snapshot_date - pd.Timedelta(days=30)
recent_30d = (
    df[df['InvoiceDate'] >= cutoff_30d]
    .groupby('Customer ID')
    .agg(recent_30d_revenue=('Revenue', 'sum'))
    .reset_index()
)

revenue_features = (
    revenue_features
    .merge(rev_stats, on='Customer ID', how='left')
    .merge(recent_30d, on='Customer ID', how='left')
)
revenue_features['recent_30d_revenue'] = revenue_features['recent_30d_revenue'].fillna(0)

print(f"Revenue: {revenue_features.shape[0]}")
revenue_features.head()

Revenue: 5878


,Customer ID,total_revenue,avg_revenue_per_item,revenue_std,max_revenue,recent_30d_revenue
0,12346.0,77556.46,1.044039,22271.229481,77183.60,0.00
1,12347.0,4921.53,1.658756,315.773658,1294.32,224.82
2,12348.0,2019.40,0.744068,279.897118,892.80,0.00
3,12349.0,4428.69,2.727026,667.017261,1757.55,1757.55
4,12350.0,334.40,1.697462,NaN,334.40,0.00


### Nhóm 5: Sản phẩm (Product Diversity Features)
- `unique_products`: Số loại sản phẩm
- `favorite_product_freq`: Số lần mua sản phẩm yêu thích
- `repeat_product_ratio`: Tỷ lệ mua lại sản phẩm cũ

In [37]:
# unique_products
product_diversity = (
    df
    .groupby('Customer ID')
    .agg(unique_products=('StockCode', 'nunique'))
    .reset_index()
)

# favorite_product_freq: tần suất mua sản phẩm được mua nhiều nhất
product_counts = (
    df
    .groupby(['Customer ID', 'StockCode'])
    .size()
    .reset_index(name='product_count')
)
fav_product = (
    product_counts
    .groupby('Customer ID')['product_count']
    .max()
    .reset_index()
    .rename(columns={'product_count': 'favorite_product_freq'})
)

# repeat_product_ratio: tỷ lệ sản phẩm được mua nhiều hơn 1 lần
repeat_ratio = (
    product_counts
    .groupby('Customer ID')
    .apply(lambda x: (x['product_count'] > 1).sum() / len(x))
    .reset_index()
    .rename(columns={0: 'repeat_product_ratio'})
)

product_diversity = (
    product_diversity
    .merge(fav_product, on='Customer ID', how='left')
    .merge(repeat_ratio, on='Customer ID', how='left')
)

print(f"Product Diversity: {product_diversity.shape[0]}")
product_diversity.head()

Product Diversity: 5878


,Customer ID,unique_products,favorite_product_freq,repeat_product_ratio
0,12346.0,27,8,0.037037
1,12347.0,126,6,0.396825
2,12348.0,25,5,0.760000
3,12349.0,138,3,0.217391
4,12350.0,17,1,0.000000


### Nhóm 6: Loyalty (Customer Loyalty Features)
- `first_purchase_date`: Ngày mua đầu tiên
- `last_purchase_date`: Ngày mua cuối cùng
- `customer_lifetime_days`: Vòng đời khách hàng
- `purchase_rate`: frequency / customer_lifetime_days

In [38]:
customer_loyalty = (
    df
    .groupby('Customer ID')
    .agg(
        first_purchase_date=('InvoiceDate', 'min'),
        last_purchase_date=('InvoiceDate', 'max'),
        frequency=('Invoice', 'nunique'),
    )
    .reset_index()
)

customer_loyalty['customer_lifetime_days'] = (
    (customer_loyalty['last_purchase_date'] - customer_loyalty['first_purchase_date']).dt.days
)

# purchase_rate: tránh chia cho 0 khi lifetime = 0 (chỉ mua 1 ngày)
customer_loyalty['purchase_rate'] = np.where(
    customer_loyalty['customer_lifetime_days'] > 0,
    customer_loyalty['frequency'] / customer_loyalty['customer_lifetime_days'],
    np.nan
)

customer_loyalty = customer_loyalty.drop(columns=['frequency'])

print(f"Customer Loyalty: {customer_loyalty.shape[0]}")
customer_loyalty.head()

Customer Loyalty: 5878


,Customer ID,first_purchase_date,last_purchase_date,customer_lifetime_days,purchase_rate
0,12346.0,2009-12-14 08:34:00,2011-01-18 10:01:00,400,0.030000
1,12347.0,2010-10-31 14:20:00,2011-12-07 15:52:00,402,0.019900
2,12348.0,2010-09-27 14:59:00,2011-09-25 13:13:00,362,0.013812
3,12349.0,2010-04-29 13:20:00,2011-11-21 09:51:00,570,0.007018
4,12350.0,2011-02-02 16:01:00,2011-02-02 16:01:00,0,NaN


### Merge tất cả features và lưu

In [39]:
feature_tables = [
    rfm,
    purchase_behavior,
    time_based,
    revenue_features,
    product_diversity,
    customer_loyalty,
]

features = feature_tables[0]
for tbl in feature_tables[1:]:
    features = features.merge(tbl, on='Customer ID', how='outer')

print(f"Final features shape: {features.shape}")
print(f"Missing values:\n{features.isnull().sum()[features.isnull().sum() > 0]}")

features.to_parquet(FEATURES_ONLINE_RETAIL_DIR / 'features.parquet', index=False)
print("Saved to features.parquet")
features.head()

Final features shape: (5878, 29)
Missing values:
avg_days_between_orders    1622
revenue_std                1623
purchase_rate              1697
dtype: int64
Saved to features.parquet


,Customer ID,Recency,Frequency,Monetary,avg_order_value,avg_items_per_order,total_quantity,total_orders,unique_products_x,unique_purchase_days,...,revenue_std,max_revenue,recent_30d_revenue,unique_products_y,favorite_product_freq,repeat_product_ratio,first_purchase_date,last_purchase_date,customer_lifetime_days,purchase_rate
0,12346.0,326,12,77556.46,6463.038333,6190.416667,74285,12,27,8,...,22271.229481,77183.60,0.00,27,8,0.037037,2009-12-14 08:34:00,2011-01-18 10:01:00,400,0.030000
1,12347.0,2,8,4921.53,615.191250,370.875000,2967,8,126,8,...,315.773658,1294.32,224.82,126,6,0.396825,2010-10-31 14:20:00,2011-12-07 15:52:00,402,0.019900
2,12348.0,75,5,2019.40,403.880000,542.800000,2714,5,25,5,...,279.897118,892.80,0.00,25,5,0.760000,2010-09-27 14:59:00,2011-09-25 13:13:00,362,0.013812
3,12349.0,19,4,4428.69,1107.172500,406.000000,1624,4,138,4,...,667.017261,1757.55,1757.55,138,3,0.217391,2010-04-29 13:20:00,2011-11-21 09:51:00,570,0.007018
4,12350.0,310,1,334.40,334.400000,197.000000,197,1,17,1,...,NaN,334.40,0.00,17,1,0.000000,2011-02-02 16:01:00,2011-02-02 16:01:00,0,NaN
